# 03 — Exploratory Data Analysis

Investigates demand, revenue, operations and customer-behavior patterns
in `trips_analysis_ready.csv`. Every chart is saved to `images/eda/` so it
can be embedded in reports/README without re-running the notebook.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path

sns.set_theme(style="whitegrid")
PROCESSED_DIR = Path("../data/processed")
IMAGES_DIR = Path("../images/eda")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

trips = pd.read_csv(PROCESSED_DIR / "trips_analysis_ready.csv", parse_dates=["request_datetime"])
completed = trips.loc[trips.trip_status == "Completed"].copy()
print(trips.shape, completed.shape)

def savefig(name):
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / name, dpi=140, bbox_inches="tight")
    plt.close()

(45000, 40) (35507, 40)


## Demand — trips by hour of day

In [2]:
plt.figure(figsize=(10, 5))
hourly = trips.groupby("request_hour").size()
sns.barplot(x=hourly.index, y=hourly.values, color="#4C72B0")
plt.title("Ride Requests by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Trips")
savefig("01_trips_by_hour.png")

## Demand — trips by day of week

In [3]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plt.figure(figsize=(9, 5))
daily = trips.groupby("request_day").size().reindex(day_order)
sns.barplot(x=daily.index, y=daily.values, color="#55A868")
plt.title("Ride Requests by Day of Week")
plt.xlabel("")
plt.ylabel("Trips")
plt.xticks(rotation=30)
savefig("02_trips_by_day.png")

## Demand — trips by city

In [4]:
plt.figure(figsize=(9, 5))
city_trips = trips.groupby("pickup_city").size().sort_values(ascending=False)
sns.barplot(x=city_trips.index, y=city_trips.values, color="#C44E52")
plt.title("Ride Requests by Pickup City")
plt.ylabel("Trips")
plt.xlabel("")
savefig("03_trips_by_city.png")

## Demand — weekend vs weekday hourly pattern

In [5]:
plt.figure(figsize=(10, 5))
pattern = trips.groupby(["is_weekend", "request_hour"]).size().unstack(0)
pattern.columns = ["Weekday", "Weekend"]
pattern.plot(ax=plt.gca())
plt.title("Weekday vs Weekend Hourly Demand")
plt.xlabel("Hour")
plt.ylabel("Trips")
savefig("04_weekday_vs_weekend.png")

## Revenue — distribution

In [6]:
plt.figure(figsize=(9, 5))
sns.histplot(completed.fare_amount, bins=50, color="#8172B2", kde=True)
plt.title("Fare Amount Distribution (Completed Trips)")
plt.xlabel("Fare (Rs )")
savefig("05_fare_distribution.png")

## Revenue — by city and vehicle type

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rev_city = completed.groupby("pickup_city").fare_amount.sum().sort_values(ascending=False)
sns.barplot(x=rev_city.index, y=rev_city.values, ax=axes[0], color="#4C72B0")
axes[0].set_title("Revenue by City")
axes[0].set_ylabel("Revenue (Rs )")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

rev_vehicle = completed.groupby("vehicle_type").fare_amount.sum().sort_values(ascending=False)
sns.barplot(x=rev_vehicle.index, y=rev_vehicle.values, ax=axes[1], color="#DD8452")
axes[1].set_title("Revenue by Vehicle Type")
axes[1].set_ylabel("Revenue (Rs )")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
savefig("06_revenue_by_city_vehicle.png")

## Operations — driver performance (top 15 by revenue)

In [8]:
plt.figure(figsize=(9, 6))
top_drivers = completed.groupby("driver_id").fare_amount.sum().sort_values(ascending=False).head(15)
sns.barplot(x=top_drivers.values, y=top_drivers.index.astype(str), orient="h", color="#55A868")
plt.title("Top 15 Drivers by Revenue")
plt.xlabel("Revenue (Rs )")
plt.ylabel("Driver ID")
savefig("07_top_drivers.png")

## Cancellations — reasons and rate by city

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
reasons = trips.loc[trips.trip_status != "Completed", "cancellation_reason"].value_counts().head(10)
sns.barplot(x=reasons.values, y=reasons.index, orient="h", ax=axes[0], color="#C44E52")
axes[0].set_title("Top Cancellation Reasons")
axes[0].set_xlabel("Trips")

cancel_by_city = trips.groupby("pickup_city").is_cancelled.mean().sort_values(ascending=False) * 100
sns.barplot(x=cancel_by_city.index, y=cancel_by_city.values, ax=axes[1], color="#937860")
axes[1].set_title("Cancellation Rate by City")
axes[1].set_ylabel("Cancellation Rate (%)")
savefig("08_cancellation_reasons_by_city.png")

## Cancellations — rate by hour and by surge bucket

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cancel_by_hour = trips.groupby("request_hour").is_cancelled.mean() * 100
sns.lineplot(x=cancel_by_hour.index, y=cancel_by_hour.values, ax=axes[0], marker="o", color="#C44E52")
axes[0].set_title("Cancellation Rate by Hour")
axes[0].set_ylabel("Cancellation Rate (%)")

trips["surge_bucket"] = pd.cut(
    trips.surge_multiplier, bins=[1.0, 1.1, 1.3, 1.5, 2.0, 3.01],
    labels=["1.0-1.1", "1.1-1.3", "1.3-1.5", "1.5-2.0", "2.0+"], include_lowest=True
)
cancel_by_surge = trips.groupby("surge_bucket", observed=True).is_cancelled.mean() * 100
sns.barplot(x=cancel_by_surge.index, y=cancel_by_surge.values, ax=axes[1], color="#4C72B0")
axes[1].set_title("Cancellation Rate by Surge Multiplier")
axes[1].set_ylabel("Cancellation Rate (%)")
savefig("09_cancellation_by_hour_surge.png")

## Customer behavior — trip frequency distribution

In [11]:
plt.figure(figsize=(9, 5))
rider_freq = completed.groupby("rider_id").size()
sns.histplot(rider_freq, bins=30, color="#8172B2")
plt.title("Rider Trip Frequency Distribution")
plt.xlabel("Completed Trips per Rider")
savefig("10_rider_frequency.png")

## Correlation heatmap (numeric trip features)

In [12]:
plt.figure(figsize=(8, 6))
num_cols = ["distance_km", "duration_min", "fare_amount", "surge_multiplier",
            "fare_per_km", "driver_avg_rating", "request_hour"]
corr = completed[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Trip Features")
savefig("11_correlation_heatmap.png")

print("EDA charts saved to images/eda/")

EDA charts saved to images/eda/
